# Bronze to Silver (upsert / rerun-safe)
Copy of `bronze_to_silver` with three changes (branch `bronze_to_silver_upsert_only`):
* Tables in `UPSERT_TABLES` (currently `lab_results`) are always MERGEd on their key, whatever `load_type` ADF sends. Within a batch the latest `reported_at` wins, and an existing silver row is only updated when the incoming `reported_at` is the same or newer.
* `APPEND` replaces the rows of the current `landing_timestamp` batch instead of inserting them again, so reruns don't create duplicates.
* Optional widget `silver_schema` (default `silver`) so the notebook can be tested against `silver_test`.

`landing_timestamp` is used only as a batch label, never for ordering: ADF writes it with a 12-hour clock.

In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("merge_key", "", "Merge Key")
dbutils.widgets.text("load_type", "", "Load Type")
dbutils.widgets.text("silver_schema", "silver", "Silver Schema")

import time
from datetime import datetime

_run_start = time.time()

# ---------- Parameters ----------
landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys") or dbutils.widgets.get("merge_key")
load_type = dbutils.widgets.get("load_type")
silver_schema = dbutils.widgets.get("silver_schema").strip() or "silver"

print("=" * 70)
print(f"[START] bronze_to_silver_upsert started at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"[PARAMS] table_name        = '{table_name}'")
print(f"[PARAMS] landing_timestamp = '{landing_timestamp}'")
print(f"[PARAMS] load_type         = '{load_type}'")
print(f"[PARAMS] merge_keys        = '{merge_keys}'")
print(f"[PARAMS] silver_schema     = '{silver_schema}'")
print("=" * 70)

if not table_name:
    print("[WARN] table_name is empty")
if not landing_timestamp:
    print("[WARN] landing_timestamp is empty - no bronze rows will match")
if not load_type:
    print("[WARN] load_type is empty - the write step will fail")

In [0]:
bronze_table = f"adb_healthcare.bronze.{table_name}"
print(f"[BRONZE] Reading batch from {bronze_table}")

bronze_query = f"""
SELECT *
FROM {bronze_table}
WHERE landing_timestamp = '{landing_timestamp}'
"""
print("[BRONZE] Query prepared:")
print("-" * 70)
print(bronze_query.strip())
print("-" * 70)

bronze_df = spark.sql(bronze_query)

bronze_count = bronze_df.count()
print(f"[BRONZE] Rows in batch : {bronze_count}")
print(f"[BRONZE] Columns       : {bronze_df.columns}")
if bronze_count == 0:
    # No new batch (e.g. ADF copied no files). Stop here so a FULL load can't overwrite silver with an empty batch.
    print(f"[SKIP] No rows found in {bronze_table} for landing_timestamp '{landing_timestamp}' - nothing new to load. Silver left unchanged.")
    print(f"[END] bronze_to_silver_upsert finished for table '{table_name}' in {time.time() - _run_start:.1f}s")
    print("=" * 70)
    dbutils.notebook.exit("0")

In [0]:
from pyspark.sql.functions import current_timestamp
updated_df = bronze_df.withColumn("insert_timestamp", current_timestamp()).withColumn("update_timestamp", current_timestamp())
print("[TRANSFORM] Set insert_timestamp and update_timestamp to current_timestamp()")
print(f"[TRANSFORM] Output columns: {updated_df.columns}")

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, col

#in silver we keep only the final updated version of the record
target_table = f"adb_healthcare.{silver_schema}.{table_name}"
source_view = f"src_{table_name}_batch"
load_type_normalized = load_type.strip().upper()

if load_type_normalized not in ("FULL", "APPEND", "MERGE"):
    print(f"[ERROR] Unsupported load_type '{load_type}'")
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected one of: FULL, APPEND, MERGE")

# Tables that are always upserted on their business key, whatever load_type ADF sends.
#   keys     : merge keys used when ADF does not pass merge_keys
#   order_by : source column that tells which version of a record is the latest
#              (landing_timestamp is not used for ordering: ADF writes it with a 12-hour clock)
UPSERT_TABLES = {
    "lab_results": {"keys": "lab_result_id", "order_by": "reported_at"},
}
upsert_cfg = UPSERT_TABLES.get(table_name)
order_by = None
if upsert_cfg:
    if load_type_normalized != "MERGE":
        print(f"[WRITE] '{table_name}' is an upsert-only table: overriding load_type {load_type_normalized} -> MERGE")
        load_type_normalized = "MERGE"
    if not merge_keys:
        merge_keys = upsert_cfg["keys"]
        print(f"[WRITE] merge_keys not provided - using default '{merge_keys}'")
    order_by = upsert_cfg["order_by"]

print(f"[WRITE] Target table : {target_table}")
print(f"[WRITE] Load type    : {load_type_normalized}")

target_exists_before = spark.catalog.tableExists(target_table)
target_count_before = spark.read.table(target_table).count() if target_exists_before else 0
print(f"[WRITE] Target exists      : {target_exists_before}")
print(f"[WRITE] Target rows before : {target_count_before}")

cols = updated_df.columns
col_list = ", ".join(f"`{c}`" for c in cols)

def run_sql(step, sql_text):
    """Print the query, execute it, and print how long it took."""
    print(f"[{step}] Query prepared:")
    print("-" * 70)
    print(sql_text.strip())
    print("-" * 70)
    t0 = time.time()
    result = spark.sql(sql_text)
    print(f"[{step}] Query finished in {time.time() - t0:.1f}s")
    return result

if load_type_normalized == "MERGE":
    # Merge: insert new records, update existing ones based on merge keys
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    print(f"[MERGE] Merge keys: {keys}")
    if not keys:
        raise ValueError("merge_keys must be provided (comma-separated) when load_type is MERGE")

    missing_keys = [k for k in keys if k not in cols]
    if missing_keys:
        raise ValueError(f"Merge keys not found in source data: {missing_keys}")
    if order_by and order_by not in cols:
        raise ValueError(f"Ordering column '{order_by}' not found in source data")

    # A merge fails if multiple source rows match the same target row, so keep one row per key
    if order_by:
        # keep the latest version of each record (by order_by)
        latest_first = Window.partitionBy(*keys).orderBy(col(order_by).desc_nulls_last())
        source_df = (updated_df
                     .withColumn("_rn", row_number().over(latest_first))
                     .filter(col("_rn") == 1)
                     .drop("_rn"))
        print(f"[MERGE] De-duplicating batch on {keys}, keeping latest '{order_by}'")
    else:
        source_df = updated_df.dropDuplicates(keys)
    source_count = source_df.count()
    duplicates_removed = bronze_count - source_count
    print(f"[MERGE] Source rows after de-duplication on keys: {source_count}")
    if duplicates_removed > 0:
        print(f"[WARN] Removed {duplicates_removed} duplicate rows on merge keys")
else:
    source_df = updated_df

# Register the prepared batch as a temp view so it can be used in SQL
source_df.createOrReplaceTempView(source_view)
print(f"[WRITE] Registered batch as temp view `{source_view}`")

if load_type_normalized == "FULL" or (load_type_normalized in ("MERGE", "APPEND") and not target_exists_before):
    # Full load (or first MERGE/APPEND run): replace / create the target table from the current batch
    if load_type_normalized in ("MERGE", "APPEND"):
        print(f"[{load_type_normalized}] Target does not exist - creating it from the batch")
    else:
        print("[FULL] Overwriting target table with the current batch")
    write_query = f"""
CREATE OR REPLACE TABLE {target_table} AS
SELECT {col_list}
FROM {source_view}
"""
    run_sql(load_type_normalized, write_query)

elif load_type_normalized == "APPEND":
    # Rerun-safe append: atomically replace the rows of this landing_timestamp batch with the current batch.
    # First run inserts the batch; a rerun of the same batch swaps it for itself (no duplicates). Other batches are untouched.
    existing_batch_rows = spark.sql(f"SELECT count(*) AS n FROM {target_table} WHERE landing_timestamp = '{landing_timestamp}'").collect()[0]["n"]
    print(f"[APPEND] Rows already in target for this batch: {existing_batch_rows} (will be replaced)")
    write_query = f"""
INSERT INTO {target_table}
REPLACE WHERE landing_timestamp = '{landing_timestamp}'
SELECT {col_list}
FROM {source_view}
"""
    run_sql("APPEND", write_query)

else:
    # MERGE into existing table
    merge_condition = "\n   AND ".join(f"t.`{k}` <=> s.`{k}`" for k in keys)
    # Only update when the incoming record is the same or newer, so rerunning an old batch can't overwrite newer data
    if order_by:
        matched_clause = f"WHEN MATCHED AND (t.`{order_by}` IS NULL OR s.`{order_by}` >= t.`{order_by}`) THEN UPDATE SET"
    else:
        matched_clause = "WHEN MATCHED THEN UPDATE SET"
    # On update, keep the original insert_timestamp and refresh everything else
    update_set = ",\n    ".join(f"t.`{c}` = s.`{c}`" for c in cols if c != "insert_timestamp")
    insert_values = ", ".join(f"s.`{c}`" for c in cols)
    write_query = f"""
MERGE INTO {target_table} AS t
USING {source_view} AS s
    ON {merge_condition}
{matched_clause}
    {update_set}
WHEN NOT MATCHED THEN INSERT ({col_list})
    VALUES ({insert_values})
"""
    merge_result = run_sql("MERGE", write_query).collect()
    if merge_result:
        m = merge_result[0].asDict()
        print(f"[MERGE] Rows affected : {m.get('num_affected_rows')}")
        print(f"[MERGE] Rows updated  : {m.get('num_updated_rows')}")
        print(f"[MERGE] Rows inserted : {m.get('num_inserted_rows')}")
        if order_by and m.get('num_updated_rows') is not None and m.get('num_inserted_rows') is not None:
            skipped = source_count - m['num_updated_rows'] - m['num_inserted_rows']
            print(f"[MERGE] Rows skipped (older '{order_by}' than silver): {skipped}")

target_count_after = spark.read.table(target_table).count()
print(f"[WRITE] Target rows after  : {target_count_after}")
print(f"[WRITE] Net row change     : {target_count_after - target_count_before}")

last_op = spark.sql(f"DESCRIBE HISTORY {target_table} LIMIT 1").select("version", "timestamp", "operation").collect()[0]
print(f"[WRITE] Delta version {last_op['version']} | {last_op['operation']} | {last_op['timestamp']}")
print(f"[WRITE] {load_type_normalized} load to {target_table} completed successfully")

In [0]:
records_count = updated_df.count()
print(f"[EXIT] Returning records_count = {records_count} to caller")
print(f"[END] bronze_to_silver_upsert finished for table '{table_name}' in {time.time() - _run_start:.1f}s")
print("=" * 70)
dbutils.notebook.exit(str(records_count))